In [ ]:
import speech_recognition as sr
import os
import pyttsx3
import webbrowser
import datetime
import PIL.ImageGrab
import anthropic

#Claude API
# Claude API key: read from an environment variable so it is never written in the code.
# Set it before starting Jupyter (see README). If it is missing, ask for it (hidden input).
import getpass
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Enter your Anthropic API key: ")
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def ask_claude(user_text):
    #Send unrecognised commands to Claude Haiku and speak the reply.
    try:
        response = claude_client.messages.create(
            model="claude-haiku-4-5-20251001", #whay this model 
            max_tokens=300, #
            system=(  
                "You are a helpful voice assistant. "
                "Keep replies short, clear, and spoken-friendly — no markdown, "
                "no bullet points, no special characters."
            ),
            messages=[{"role": "user", "content": user_text}]
        )
        return response.content[0].text
    except Exception as e:
        return f"Sorry, I couldn't reach the AI. Error: {str(e)}"

#TTS engine
# first runAndWait(). Reinitializing each call is the reliable workaround.
def assistant_talk(text):
    print("Assistant:", text)
    engine = pyttsx3.init()
    engine.setProperty('rate', 170) # 170 words per minuts 
    engine.setProperty('volume', 1.0)
    voices = engine.getProperty('voices')
    engine.setProperty('voice', voices[1].id)  # 0 = male, 1 = female
    engine.say(text)
    engine.runAndWait()
    engine.stop()

#Listen (initialized once — no startup delay per call)
recognizer = sr.Recognizer()
mic        = sr.Microphone()

def calibrate_microphone():
    #Call once at startup to avoid per-call ambient noise delay
    print("Calibrating microphone...")
    with mic as source:
        recognizer.adjust_for_ambient_noise(source, duration=1)
    print("Microphone ready.")
# open micrphone record until the user stop speaking then send it to google speech recognition api and return text and print it as user(you) and assistant
def assistant_listen():
    with mic as source:
        print("Listening...")
        audio = recognizer.listen(source)
        #handle with error 
    try:
        text = recognizer.recognize_google(audio)
        print("You:", text)
        return text.lower()
    except sr.UnknownValueError:  #audio recived but can't convert it to text 
        assistant_talk("Sorry, I didn't understand.")
        return ""
    except sr.RequestError:  # network or api error
        assistant_talk("Network error.")
        return ""

# Screenshot
def get_desktop_path():
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    if os.path.exists(desktop):
        return desktop
    onedrive_desktop = os.path.join(os.path.expanduser("~"), "OneDrive", "Desktop")
    if os.path.exists(onedrive_desktop):
        return onedrive_desktop
    documents = os.path.join(os.path.expanduser("~"), "Documents")
    if os.path.exists(documents):
        return documents
    return os.path.expanduser("~")

def take_screenshot():
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename  = f"screenshot_{timestamp}.png"
    folder    = get_desktop_path()
    filepath  = os.path.join(folder, filename)
    screenshot = PIL.ImageGrab.grab()
    screenshot.save(filepath)
    assistant_talk(f"Screenshot saved as {filename}")
    print(f"[Saved to]: {filepath}")

#Automation commands (10-second delay user can cancle)
def handle_shutdown():
    assistant_talk("Shutting down your computer in 10 seconds. Goodbye!")
    os.system("shutdown /s /t 10") 

def handle_restart():
    assistant_talk("Restarting your computer in 10 seconds. See you soon!")
    os.system("shutdown /r /t 10")

def handle_lock():
    assistant_talk("Locking your screen.")
    os.system("rundll32.exe user32.dll,LockWorkStation")

def handle_sleep():
    assistant_talk("Putting your computer to sleep. Good night!")
    os.system("rundll32.exe powrprof.dll,SetSuspendState 0,1,0")

#Main command router
def run_assistant():
    text = assistant_listen()
    if text == "":
        return

    words = set(text.split())   # word-level match — handles missed first words

    # Greetings 
    if words & {"hello", "hi", "hey"}:
        assistant_talk("Hello! How can I help you?")

    # Time & Date
    elif "time" in words and "timer" not in words:
        time_now = datetime.datetime.now().strftime("%I:%M %p")
        assistant_talk(f"The time is {time_now}")

    elif "date" in words or ("day" in words and "today" in words):
        date_today = datetime.datetime.now().strftime("%B %d %Y")
        assistant_talk(f"Today is {date_today}")

    # Web
    elif "google" in words:
        webbrowser.open("https://www.google.com")
        assistant_talk("Opening Google")

    elif "youtube" in words:
        webbrowser.open("https://www.youtube.com")
        assistant_talk("Opening YouTube")

    elif "search" in words:
        query = text.replace("search", "").strip()
        webbrowser.open(f"https://www.google.com/search?q={query}")
        assistant_talk(f"Searching for {query}")

    #Apps
    elif "word" in words:
        os.system("start winword")
        assistant_talk("Opening Word")

    elif "notepad" in words:
        os.system("notepad")
        assistant_talk("Opening Notepad")

    elif "calculator" in words:
        os.system("calc")
        assistant_talk("Opening Calculator")

    #Screenshot 
    elif "screenshot" in words:
        take_screenshot()

    # Automation 
    elif "cancel" in words or "abort" in words:
        os.system("shutdown /a")
        assistant_talk("Shutdown cancelled.")

    elif words & {"shutdown", "off"} and "turn" in words or \
         "shutdown" in words or "poweroff" in text.replace(" ", ""):
        handle_shutdown()

    elif "restart" in words or "reboot" in words:
        handle_restart()

    elif "lock" in words:
        handle_lock()

    elif "sleep" in words or "hibernate" in words:
        handle_sleep()

    #Exit assistant 
    elif "exit" in words or "stop" in words or "quit" in words:
        assistant_talk("Goodbye!")
        exit()

    # Unknown ask Claude
    else:
        assistant_talk("Let me think about that.")
        reply = ask_claude(text)
        assistant_talk(reply)

#  Entry point
if __name__ == "__main__":
    calibrate_microphone()
    assistant_talk("Assistant started. Say something.")
    while True:
        run_assistant()